## Load the extension

In [1]:
%load_ext autoreload
%aimport -sparkmagic # it loses the references to the sessions if it reloads
%autoreload 2

In [2]:
%load_ext livy_uploads.magics

## Fetching remote variable

In [3]:
print(list(sorted(globals())))

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
7,None,pyspark,idle,,,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

['HiveContext', 'StreamingContext', '__builtins__', 'cloudpickle', 'sc', 'spark', 'sqlContext']

In [4]:
%local

print(list(sorted(globals())))

['In', 'Out', '_', '__', '___', '__builtin__', '__builtins__', '__doc__', '__loader__', '__name__', '__package__', '__spec__', '_dh', '_i', '_i1', '_i2', '_i3', '_i4', '_ih', '_ii', '_iii', '_oh', 'display_dataframe', 'exit', 'get_ipython', 'ip', 'quit']


In [5]:
from datetime import datetime

now = datetime.now().astimezone()
now

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

datetime.datetime(2025, 9, 2, 2, 18, 51, 667621, tzinfo=datetime.timezone(datetime.timedelta(0), 'UTC'))

In [6]:
%get_obj_from_spark -n now

In [7]:
%local

now

datetime.datetime(2025, 9, 2, 2, 18, 51, 667621, tzinfo=datetime.timezone(datetime.timedelta(0), 'UTC'))

In [8]:
%local

from datetime import datetime

delta = (datetime.now().astimezone() - now)
assert delta.total_seconds() < 30

## Sending local variable

In [9]:
%local

foo = {2, 3, 4}


In [10]:
%send_obj_to_spark -n foo

In [11]:
foo_total = sum(foo)

assert foo == {2, 3, 4}

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [12]:
%get_obj_from_spark -n foo_total

In [13]:
%local

assert foo_total == 9

## Running commands

In [14]:
%%shell_command

ls -lahF .

total 348K
drwxrwxrwt  1 root root 4.0K Sep  2 02:18 ./
drwxr-xr-x  1 root root 4.0K Sep  2 02:07 ../
-rw-------  1 app  app   22K Sep  2 02:07 6078788278198361295
-rw-------  1 app  app   22K Sep  2 02:18 7645205307348949484
drwxr-xr-x  2 app  app  4.0K Sep  2 02:18 blockmgr-7eef8070-bc3d-4dcc-8252-df987f19c43f/
drwxr-xr-x  4 app  app  4.0K Sep  2 02:09 blockmgr-7fd8bcfa-f450-4059-9ed5-0445ea3c45b7/
drwxr-xr-x  2 app  app  4.0K Sep  2 02:18 hsperfdata_app/
drwxr-xr-x  2 root root 4.0K Oct 19  2024 hsperfdata_root/
-rw-r--r--  1 app  app  199K Sep  2 02:09 liblz4-java-4413688502684583384.so
-rw-r--r--  1 app  app     0 Sep  2 02:09 liblz4-java-4413688502684583384.so.lck
-rw-------  1 app  app  3.9K Sep  2 02:18 livyConf7238440442894544375.properties
-rw-------  1 app  app  3.9K Sep  2 02:07 livyConf735663374858564102.properties
drwx------  2 app  app  4.0K Sep  2 02:18 rsc-tmp4279860722890942843/
drwx------  2 app  app  4.0K Sep  2 02:07 rsc-tmp7696244115896949586/
drwxr-xr-x  2 app  a

In [15]:
%%shell_command

bash -c 'echo foo && exit 42'

foo
$ command exited with code 42

In [16]:
%%local

assert shell_output == 'foo\n'
assert shell_returncode == 42


## Sending local file

In [17]:
%local !ls -lahF

total 104K
drwxrwxr-x  5 app app 4.0K Aug 30 15:12 ./
drwxrwxr-x 19 app app 4.0K Sep  2 02:10 ../
drwxr-xr-x  2 app app 4.0K Jan 13  2025 .ipynb_checkpoints/
-rw-rw-r--  1 app app  48K Aug 30 15:12 magics.ipynb
drwxrwxr-x  3 app app 4.0K Jan  9  2025 sample-dir/
drwxrwxr-x  2 app app 4.0K Aug 30 15:12 spark/
-rw-rw-r--  1 app app  33K Aug 30 15:12 test-spark-another-version.ipynb


In [18]:
%send_path_to_spark -p magics.ipynb

Uploaded magics.ipynb to /tmp/magics.ipynb

In [19]:
%%shell_command

ls -lahF | grep magics

-rw-------  1 app  app   48K Sep  2 02:19 magics.ipynb
$ command exited with code 0

In [20]:
%%local

assert 'magics.ipynb' in shell_output

## Sending local directory

In [21]:
%local !find sample-dir/

sample-dir/
sample-dir/inner
sample-dir/inner/bar.txt
sample-dir/foo.txt


In [22]:
%send_path_to_spark -p sample-dir/

Uploaded sample-dir to /tmp/sample-dir

In [23]:
%%shell_command
pwd

/tmp
$ command exited with code 0

In [24]:
%%shell_command

find "$PWD/sample-dir"

/tmp/sample-dir
/tmp/sample-dir/inner
/tmp/sample-dir/inner/bar.txt
/tmp/sample-dir/foo.txt
$ command exited with code 0

In [25]:
%%local

assert 'sample-dir/' in shell_output

## Following session logs

In [26]:
%logs_follow -p 500

stdout: 

stderr: 
25/09/02 02:18:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/09/02 02:18:44 INFO RSCDriver: Connecting to: livy.docker.internal:10000
25/09/02 02:18:44 INFO RSCDriver: Starting RPC server...
25/09/02 02:18:44 INFO RpcServer: Connected to the port 10002
25/09/02 02:18:44 WARN ClientConf: Your hostname, livy, resolves to a loopback address, but we couldn't find any external IP address!
25/09/02 02:18:44 WARN ClientConf: Set livy.rsc.rpc.server.address if you need to bind to another address.
25/09/02 02:18:44 INFO RSCDriver: Received job request 998de32c-21cb-46fa-8459-1888d3a51d66
25/09/02 02:18:44 INFO RSCDriver: SparkContext not yet up, queueing job request.
25/09/02 02:18:46 INFO SparkEntries: Starting Spark context...
25/09/02 02:18:46 INFO SparkContext: Running Spark version 3.2.1
25/09/02 02:18:46 INFO ResourceUtils: ==============================================================


In [27]:
%logs_follow -p 500

No new logs

In [28]:
sc._gateway.jvm.java.lang.System.err.println('Hello World')

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [29]:
%logs_follow -p 500

Hello World


In [30]:
%local

assert 'Hello World' in '\n'.join(logs_lines)